<a href="https://colab.research.google.com/github/dkayikci/COINS/blob/main/MetaHybridsGeo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================================
# META-LEARNING HYBRIDS — CORRECTED RE-RUN (Heritage revision)
# MAML + ResNet18 + signal and Prototypical Networks + EfficientNet-B0 + signal,
# Stage 1 and Stage 2, overlapping-test and held-out designs, balanced and legacy layouts.
#
# Taken verbatim from the main pipeline (e4fix): signal descriptors (aligned 112/200/175 and
# legacy blocks), budget allocation (D = 200 -> 80/60/60), SignalNet (trunk, signal branch
# 200-128-64, head 256), the Stage 1 rule (trunk[-1] trainable, earlier stages frozen),
# the splits (train_test_split, 0.2, stratified, random_state = SEED + fold) and the scaler
# (StandardScaler fitted on the training partition of each fold).
#
# Corrections relative to the original meta-learning notebooks
#  1. Stage 1 / Stage 2 exactly as in the main pipeline (trainable parameters asserted).
#  2. MAML: adapted weights are actually used (second-order MAML via torch.func.functional_call);
#     at test time the meta-learned initialisation is adapted on the training partition and the
#     adapted weights classify the test images.
#  3. ProtoNet: test images classified by the nearest class prototype; no separate softmax head.
#  4. Signal scaler fitted on the training partition only.
#  5. No explicit L2 penalty on all parameters; weight decay only through the optimiser.
#  6. Inner-loop settings (inner learning rate, inner steps) are part of the 8-combination search.
#
# Results are saved to Drive after every run; re-running skips finished runs.
# =====================================================================================

# ===================== CONFIG — edit this block only =====================
TASK     = "geo"                     # "chro" or "geo"
METHODS  = ["maml", "proto"]          # add "plain" for the optional sanity check (main-pipeline hybrid)
PLAIN_BACKBONE = "resnet18"           # backbone for "plain"
STAGES   = [1, 2]                     # 1 = frozen layers (trunk[-1] trainable), 2 = full fine-tuning
DESIGNS  = ["overlap", "heldout"]
LAYOUTS  = ["balanced", "legacy"]
ROOT = {"chro": "/content/drive/MyDrive/UABandThesis/ThesisPhD/Original_Codes_Chapter5/MetaCoinsChro/",
        "geo":  "/content/drive/MyDrive/UABandThesis/ThesisPhD/Original_Codes_Chapter5/MetaCoinsGeo/"}[TASK]
OUT_DIR = "/content/drive/MyDrive/UABandThesis/ThesisPhD/Original_Codes_Chapter5/meta_rerun_results"
SECOND_ORDER = True

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time, random, platform, itertools, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset
from torch.func import functional_call
import torchvision
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
import scipy, scipy.signal, scipy.stats
import pywt
from scipy.signal import hilbert
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
warnings.filterwarnings('ignore')

SEED = 123; IMG = 224; BATCH = 8; FOLDS = 4; FIXED_DIM = 200
UNFREEZE_LAST_N = 1
PROPS = (0.40, 0.30, 0.30)
GRID_PLAIN = [{"lr": lr, "wd": wd, "epochs": ep, "patience": 3}
              for lr in (0.001, 0.002) for wd in (0.0005, 0.001) for ep in (15, 20)]
ABL_HP = {"lr": 0.001, "wd": 0.0005, "epochs": 20, "patience": 3}      # main pipeline, held-out design
PARAM_REF = {("resnet18", 1): 8575938, ("resnet18", 2): 11358722,
             ("efficientnet_b0", 1): 790978, ("efficientnet_b0", 2): 4386366}

os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {"python": platform.python_version(), "torch": torch.__version__,
       "torchvision": torchvision.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print(ENV)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(SEED)

# ===================== SIGNAL DESCRIPTORS (verbatim from the main pipeline) =====================
NATIVE = {"wavelet": 112, "hht": 200, "laws": 175}
LAWS_1D = {"L5": np.array([1,4,6,4,1]), "E5": np.array([-1,-2,0,2,1]),
           "S5": np.array([-1,0,2,0,-1]), "W5": np.array([-1,2,0,-2,1]),
           "R5": np.array([1,-4,6,-4,1])}
LAWS_NAMES = [f"{a}x{b}" for a in LAWS_1D for b in LAWS_1D]

def _entropy(x):
    p = np.abs(np.asarray(x)).flatten(); t = p.sum()
    if t <= 0: return 0.0
    p = p/t; p = p[p > 0]
    return float(-np.sum(p*np.log2(p)))

def _stats7(v):
    v = np.asarray(v).flatten()
    return [float(np.mean(np.abs(v))), float(np.std(v)), float(np.var(v)),
            float(np.mean(v**2)), _entropy(v),
            float(scipy.stats.skew(v)), float(scipy.stats.kurtosis(v))]

def _emd(sig, n=5):
    imfs=[]; res=np.asarray(sig, dtype=float).copy()
    for _ in range(n):
        if len(res) < 10: break
        pk = scipy.signal.find_peaks(res)[0]; vl = scipy.signal.find_peaks(-res)[0]
        if len(pk) < 2 or len(vl) < 2: break
        up = np.interp(range(len(res)), pk, res[pk]); lo = np.interp(range(len(res)), vl, res[vl])
        imf = res - (up+lo)/2; imfs.append(imf); res = res - imf
        if np.std(imf) < 0.01: break
    return imfs

def sampled_line_indices(gray, axis, n_lines=5):
    n = gray.shape[0] if axis == "row" else gray.shape[1]
    return list(range(0, n, max(1, n//10)))[:n_lines]

def wavelet_block(gray):
    f = []
    for lvl in range(1, 5):
        c = pywt.wavedec2(gray, "db4", level=lvl)
        for band in (c[0], c[1][0], c[1][1], c[1][2]):
            f += _stats7(band)
    return np.array(f)

def hht_block(gray, n_imfs=5, n_lines=5):
    f = []
    for axis in ["row","col"]:
        idx = sampled_line_indices(gray, axis, n_lines)
        for li in range(n_lines):
            if li < len(idx):
                imfs = _emd(gray[idx[li],:] if axis=="row" else gray[:,idx[li]], n_imfs)
            else:
                imfs = []
            for k in range(n_imfs):
                if k < len(imfs):
                    an = hilbert(imfs[k]); a = np.abs(an)
                    fr = np.diff(np.unwrap(np.angle(an)))/(2*np.pi)
                    f += [float(np.mean(a)), float(np.std(a)),
                          float(np.mean(fr)) if len(fr) else 0.0,
                          float(np.std(fr))  if len(fr) else 0.0]
                else:
                    f += [0.0, 0.0, 0.0, 0.0]
    return np.array(f)

def laws_block(gray):
    f = []
    for name in LAWS_NAMES:
        a, b = name.split("x")
        r = scipy.signal.convolve2d(gray, np.outer(LAWS_1D[a], LAWS_1D[b]), mode="valid")
        f += _stats7(np.abs(r))
    return np.array(f)

def legacy_blocks(gray):
    c = pywt.wavedec2(gray, "db4", level=4); w=[]
    for i, coeff in enumerate(c):
        if i == 0:
            w += [np.mean(coeff), np.std(coeff), np.var(coeff),
                  scipy.stats.skew(coeff.flatten()), scipy.stats.kurtosis(coeff.flatten())]
        else:
            for d in coeff:
                w += [np.mean(d), np.std(d), np.var(d), np.max(d), np.min(d),
                      scipy.stats.skew(d.flatten()), scipy.stats.kurtosis(d.flatten())]
    h=[]
    for axis in ["row","col"]:
        for i in sampled_line_indices(gray, axis):
            for imf in _emd(gray[i,:] if axis=="row" else gray[:,i]):
                an = hilbert(imf); a = np.abs(an)
                fr = np.diff(np.unwrap(np.angle(an)))/(2*np.pi)
                h += [np.mean(a), np.std(a),
                      np.mean(fr) if len(fr) else 0, np.std(fr) if len(fr) else 0]
    l=[]
    for name in LAWS_NAMES:
        a, b = name.split("x")
        t = np.abs(scipy.signal.convolve2d(gray, np.outer(LAWS_1D[a], LAWS_1D[b]), mode="valid"))
        l += [np.mean(t), np.std(t), np.var(t), np.max(t), np.min(t),
              scipy.stats.skew(t.flatten()), scipy.stats.kurtosis(t.flatten())]
    return np.array(w), np.array(h), np.array(l)

def allocate(budget, props=PROPS, native=NATIVE):
    if budget <= 0: return {"wavelet":0, "hht":0, "laws":0}
    keys = ["wavelet","hht","laws"]
    want = {k:int(round(budget*p)) for k,p in zip(keys, props)}
    want[keys[-1]] = budget - want[keys[0]] - want[keys[1]]
    for _ in range(4):
        over = 0
        for k in keys:
            if want[k] > native[k]: over += want[k]-native[k]; want[k] = native[k]
        if over == 0: break
        room = {k: native[k]-want[k] for k in keys}
        tr = sum(room.values())
        if tr == 0: break
        for k in keys:
            want[k] += min(room[k], int(round(over*room[k]/tr)))
    return want

def fit_len(v, n):
    if n <= 0: return np.zeros(0)
    if len(v) > n: return v[:n]
    if len(v) < n: return np.concatenate([v, np.zeros(n-len(v))])
    return v

def build_vector(blocks, budget, mode="balanced"):
    w,h,l = blocks
    if budget <= 0: return np.zeros(0)
    if mode == "legacy":
        return fit_len(np.concatenate([w,h,l]), budget)
    a = allocate(budget)
    return np.concatenate([fit_len(w,a["wavelet"]), fit_len(h,a["hht"]), fit_len(l,a["laws"])])

# ===================== DATA (same loading, order and grey-level conversion as the main pipeline) =====================
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
def denorm_hwc(x): return torch.clamp(x*STD+MEAN, 0, 1).permute(1,2,0).numpy()

tfm = transforms.Compose([transforms.Resize((IMG,IMG)), transforms.ToTensor(),
                          transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))])
splits = {s: ImageFolder(os.path.join(ROOT, s), transform=tfm) for s in ["train","test"]}
classes = splits["train"].classes
images, labels, aligned, legacy, split_of, paths = [], [], [], [], [], []
t0 = time.time()
for sname, ds in splits.items():
    for i in range(len(ds)):
        x, y = ds[i]
        images.append(x); labels.append(y); split_of.append(sname)
        paths.append(f"{sname}/{os.path.basename(ds.samples[i][0])}")
        gray = np.mean(denorm_hwc(x), axis=2)
        aligned.append((wavelet_block(gray), hht_block(gray), laws_block(gray)))
        legacy.append(legacy_blocks(gray))
images = torch.stack(images); labels = np.array(labels); split_of = np.array(split_of)
nC = len(classes); N = len(labels)
test_idx = np.where(split_of == "test")[0]
print(f"Descriptor extraction: {time.time()-t0:.1f}s for {N} images")
hl = sorted({len(b[1]) for b in legacy})
print(f"CHECK aligned lengths: {[len(b) for b in aligned[0]]}  (main pipeline: [112, 200, 175])")
print(f"CHECK legacy HHT length: {hl[0]}-{hl[-1]} ({len(hl)} distinct)  (main pipeline: chro 64-104 (11), geo 56-108)")
print(f"Classes: {classes} | train {(split_of=='train').sum()} | test {len(test_idx)} | total {N}")
VEC = {m: np.array([build_vector(b, FIXED_DIM, m) for b in (legacy if m == "legacy" else aligned)])
       for m in ("balanced", "legacy")}

# ===================== MODEL (main pipeline's SignalNet; stage passed explicitly) =====================
def build_trunk(name, stage):
    if name.startswith("efficientnet"):
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT); trunk = m.features
    else:
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT); trunk = nn.Sequential(*list(m.children())[:-2])
    if stage == 1:
        for p in trunk.parameters(): p.requires_grad = False
        for i in range(len(trunk)-UNFREEZE_LAST_N, len(trunk)):
            for p in trunk[i].parameters(): p.requires_grad = True
    vdim = 1280 if name.startswith("efficientnet") else 512
    return trunk, vdim

class SignalNet(nn.Module):
    def __init__(self, num_classes, sdim, backbone, stage):
        super().__init__()
        self.features, vdim = build_trunk(backbone, stage)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.signal = nn.Sequential(nn.Linear(sdim,128), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(128,64),  nn.ReLU(), nn.Dropout(0.2))
        self.head = nn.Sequential(nn.Linear(vdim+64,256), nn.ReLU(), nn.Dropout(0.4),
                                  nn.Linear(256,num_classes))
    def embed(self, x, s):          # ProtoNet embedding = head without its final layer
        v = torch.cat([self.pool(self.features(x)).flatten(1), self.signal(s)], dim=1)
        return self.head[:3](v)
    def forward(self, x, s):
        return self.head[3](self.embed(x, s))

BACKBONE = {"maml": "resnet18", "proto": "efficientnet_b0", "plain": PLAIN_BACKBONE}

def make_model(method, stage):
    bb = BACKBONE[method]
    m = SignalNet(nC, FIXED_DIM, bb, stage)
    if method == "proto": m.head[3].requires_grad_(False)       # output layer is not used by ProtoNet
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    exp = PARAM_REF[(bb, stage)] + (nC-2)*257 - (257*nC if method == "proto" else 0)
    assert n == exp, f"{method} stage {stage}: trainable {n:,} != expected {exp:,}"
    return m.to(device), n

# ===================== SHARED HELPERS =====================
class VecDataset(Dataset):
    def __init__(self, V): self.V = V
    def __len__(self): return N
    def __getitem__(self, i): return images[i], torch.tensor(self.V[i], dtype=torch.float32), int(labels[i])
def collate(b):
    xs, ss, ys = zip(*b); return torch.stack(xs), torch.stack(ss), torch.tensor(ys)
def batch(idx, V):
    idx = np.asarray(idx)
    return (images[idx].to(device), torch.tensor(V[idx], dtype=torch.float32).to(device),
            torch.tensor(labels[idx]).to(device))

def episode(pool_idx, rng, n_shot=5, n_query=5):
    pool_idx = np.asarray(pool_idx); s, q = [], []
    for c in range(nC):
        ci = pool_idx[labels[pool_idx] == c]
        pick = rng.choice(ci, n_shot+n_query, replace=len(ci) < n_shot+n_query)
        s += list(pick[:n_shot]); q += list(pick[n_shot:])
    return np.array(s), np.array(q)

def scores(y, p):
    _, _, f1w, _ = precision_recall_fscore_support(y, p, average="weighted", zero_division=0)
    pr, rc, f1m, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    return {"acc": float(accuracy_score(y, p)), "f1_weighted": float(f1w), "f1_macro": float(f1m),
            "precision": float(pr), "recall": float(rc),
            "cm": confusion_matrix(y, p, labels=list(range(nC))).tolist()}

# ===================== MAML (second order, functional) =====================
GRID_MAML = [{"meta_lr": a, "inner_lr": b, "inner_steps": c}
             for a in (0.001, 0.002) for b in (0.005, 0.01) for c in (3, 5)]
MAML_EPISODES, MAML_TEST_EPOCHS = 60, 10

def maml_adapt(model, params, x, s, y, lr, steps, create_graph):
    p = dict(params)
    for _ in range(steps):
        loss = F.cross_entropy(functional_call(model, p, (x, s)), y)
        g = torch.autograd.grad(loss, list(p.values()), create_graph=create_graph, allow_unused=True)
        p = {k: (v - lr*gi if gi is not None else v) for (k, v), gi in zip(p.items(), g)}
    return p

def fit_maml(stage, hp, tr, V, eval_sets, seed):
    set_seed(seed); rng = np.random.default_rng(seed)
    model, n = make_model("maml", stage)
    opt = Adam([p for p in model.parameters() if p.requires_grad], lr=hp["meta_lr"], weight_decay=0.0005)
    for _ in range(MAML_EPISODES):                      # meta-training on episodes from the training partition
        s, q = episode(tr, rng); model.train()
        xs, ss, ys = batch(s, V); xq, sq, yq = batch(q, V)
        params = {k: v for k, v in model.named_parameters() if v.requires_grad}
        ad = maml_adapt(model, params, xs, ss, ys, hp["inner_lr"], hp["inner_steps"], SECOND_ORDER)
        loss = F.cross_entropy(functional_call(model, ad, (xq, sq)), yq)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    # adaptation of the meta-learned initialisation on the whole training partition, then prediction
    model.train(); rng2 = np.random.default_rng(seed + 7)
    params = {k: v.detach().clone().requires_grad_(True) for k, v in model.named_parameters() if v.requires_grad}
    for _ in range(MAML_TEST_EPOCHS):
        for b in np.array_split(rng2.permutation(np.asarray(tr)), max(1, len(tr)//BATCH)):
            x, s_, y = batch(b, V)
            params = maml_adapt(model, params, x, s_, y, hp["inner_lr"], 1, False)
            params = {k: v.detach().requires_grad_(True) for k, v in params.items()}
    model.eval(); out = []
    with torch.no_grad():
        for idx in eval_sets:
            x, s_, _ = batch(idx, V); out.append(functional_call(model, params, (x, s_)).argmax(1).cpu().numpy())
    return out, n

# ===================== PROTOTYPICAL NETWORKS =====================
GRID_PROTO = [{"lr": a, "wd": b, "episodes": c} for a in (0.001, 0.002) for b in (0.0005, 0.001) for c in (60, 100)]

def protos_of(model, idx, V):
    idx = np.asarray(idx)
    e = torch.cat([model.embed(*batch(b, V)[:2]) for b in np.array_split(idx, max(1, len(idx)//16))])
    y = torch.tensor(labels[idx]).to(device)
    return torch.stack([e[y == c].mean(0) for c in range(nC)])

def fit_proto(stage, hp, tr, V, eval_sets, seed):
    set_seed(seed); rng = np.random.default_rng(seed)
    model, n = make_model("proto", stage)
    opt = Adam([p for p in model.parameters() if p.requires_grad], lr=hp["lr"], weight_decay=hp["wd"])
    for _ in range(hp["episodes"]):
        s, q = episode(tr, rng); model.train()
        xs, ss, ys = batch(s, V); xq, sq, yq = batch(q, V)
        es = model.embed(xs, ss); eq = model.embed(xq, sq)
        P = torch.stack([es[ys == c].mean(0) for c in range(nC)])
        loss = F.cross_entropy(-torch.cdist(eq, P)**2, yq)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    model.eval(); out = []
    with torch.no_grad():
        P = protos_of(model, tr, V)                      # prototypes from the whole training partition
        for idx in eval_sets:
            x, s_, _ = batch(idx, V); out.append((-torch.cdist(model.embed(x, s_), P)).argmax(1).cpu().numpy())
    return out, n

# ===================== PLAIN HYBRID (main pipeline's train_model; optional sanity check) =====================
def fit_plain(stage, hp, tr, V, eval_sets, seed):
    set_seed(seed)
    model, n = make_model("plain", stage)
    loader = DataLoader(Subset(VecDataset(V), list(tr)), batch_size=BATCH, shuffle=True, collate_fn=collate)
    opt = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=hp["lr"], weight_decay=hp["wd"])
    crit = nn.CrossEntropyLoss(); model.train(); best = float("inf"); bad = 0
    for _ in range(hp["epochs"]):
        tot = 0; nb = 0
        for x, s, y in loader:
            x, s, y = x.to(device), s.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x, s), y); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tot += loss.item(); nb += 1
        avg = tot/nb if nb else float("inf")
        if avg < best: best, bad = avg, 0
        else:
            bad += 1
            if bad >= hp.get("patience", 3): break
    model.eval(); out = []
    with torch.no_grad():
        for idx in eval_sets:
            x, s_, _ = batch(idx, V); out.append(model(x, s_).argmax(1).cpu().numpy())
    return out, n

FIT  = {"maml": fit_maml, "proto": fit_proto, "plain": fit_plain}
GRID = {"maml": GRID_MAML, "proto": GRID_PROTO, "plain": GRID_PLAIN}

# ===================== RUNNER — splits exactly as in the main pipeline =====================
# overlap : tr, va = split(all images); inner split of tr; train on tr; test = test folder
# heldout : tr, te = the same split; train on tr; test = te (never trained on)
# The plain hybrid uses the main pipeline's fixed ABL_HP in the held-out design; the meta-learning
# hybrids are searched over their 8-combination grid in both designs.
def run(method, stage, design, layout):
    tag = f"{TASK}_{method}_stage{stage}_{design}_{layout}"
    out_path = os.path.join(OUT_DIR, f"{tag}.json")
    if os.path.exists(out_path):
        print(f"skip {tag} (already done)"); return json.load(open(out_path))
    print(f"\n{'='*70}\n{tag}\n{'='*70}"); t0 = time.time()
    fit = FIT[method]; folds = []
    for fold in range(FOLDS):
        tr, va = train_test_split(list(range(N)), test_size=0.2, stratify=labels, random_state=SEED+fold)
        itr, iva = train_test_split(tr, test_size=0.2, stratify=[labels[i] for i in tr], random_state=SEED+fold)
        V = StandardScaler().fit(VEC[layout][tr]).transform(VEC[layout])
        test = test_idx.tolist() if design == "overlap" else va
        if method == "plain" and design == "heldout":
            best, inner = ABL_HP, []
        else:
            inner = []
            for k, hp in enumerate(GRID[method]):
                (p,), n = fit(stage, hp, itr, V, [iva], SEED+fold)
                inner.append(float(accuracy_score(labels[iva], p)))
                print(f"  fold {fold+1} [{k+1}/{len(GRID[method])}] {hp} -> {inner[-1]:.4f}")
                torch.cuda.empty_cache()
            best = GRID[method][int(np.argmax(inner))]
        (p_tr, p_te), n = fit(stage, best, tr, V, [tr, test], SEED+fold)
        s_tr, s_te = scores(labels[tr], p_tr), scores(labels[test], p_te)
        ov = float(np.isin(test, tr).mean())
        print(f"  fold {fold+1}: best {best} | trainable {n:,} | test images also in training {ov:.0%} | "
              f"train {s_tr['acc']:.4f} | TEST {s_te['acc']:.4f}")
        folds.append({"fold": fold+1, "best": best, "inner_val_acc": inner, "trainable": n,
                      "test_in_train": ov, "n_test": len(test), "train": s_tr, "test": s_te,
                      "test_files": [paths[i] for i in test]})
        torch.cuda.empty_cache()
    acc = np.array([f["test"]["acc"] for f in folds]); f1 = np.array([f["test"]["f1_weighted"] for f in folds])
    res = {"tag": tag, "task": TASK, "method": method, "backbone": BACKBONE[method], "stage": stage,
           "design": design, "layout": layout, "classes": classes, "second_order": SECOND_ORDER,
           "maml_episodes": MAML_EPISODES, "maml_test_epochs": MAML_TEST_EPOCHS, "env": ENV,
           "minutes": (time.time()-t0)/60, "folds": folds,
           "acc_mean": float(acc.mean()), "acc_sd": float(acc.std()), "f1_mean": float(f1.mean()),
           "f1_sd": float(f1.std()), "pooled_cm": np.sum([f["test"]["cm"] for f in folds], axis=0).tolist()}
    json.dump(res, open(out_path, "w"), indent=1)
    print(f"==> {tag}: {100*acc.mean():.1f}% ± {100*acc.std():.1f} | F1 {f1.mean():.3f} | {res['minutes']:.1f} min")
    return res

ALL = [run(m, st, d, l) for m, st, d, l in itertools.product(METHODS, STAGES, DESIGNS, LAYOUTS)]

import pandas as pd
df = pd.DataFrame([{"run": r["tag"], "acc %": round(100*r["acc_mean"], 1), "sd": round(100*r["acc_sd"], 1),
                    "F1": round(r["f1_mean"], 3), "folds": [round(f["test"]["acc"], 3) for f in r["folds"]],
                    "in_train %": round(100*np.mean([f["test_in_train"] for f in r["folds"]])),
                    "trainable": r["folds"][0]["trainable"], "min": round(r["minutes"], 1)} for r in ALL])
print(df.to_string(index=False))
df.to_csv(os.path.join(OUT_DIR, f"summary_{TASK}.csv"), index=False)
print(ENV)

Mounted at /content/drive
{'python': '3.13.16', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'cuda': '13.0', 'gpu': 'NVIDIA A100-SXM4-40GB'}
Descriptor extraction: 54.3s for 80 images
CHECK aligned lengths: [112, 200, 175]  (main pipeline: [112, 200, 175])
CHECK legacy HHT length: 56-108 (13 distinct)  (main pipeline: chro 64-104 (11), geo 56-108)
Classes: ['Abydos', 'Dikaia', 'Kyzikos', 'Lampsakos'] | train 64 | test 16 | total 80

geo_maml_stage1_overlap_balanced
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


  fold 1 [1/8] {'meta_lr': 0.001, 'inner_lr': 0.005, 'inner_steps': 3} -> 0.9231
  fold 1 [2/8] {'meta_lr': 0.001, 'inner_lr': 0.005, 'inner_steps': 5} -> 0.9231
  fold 1 [3/8] {'meta_lr': 0.001, 'inner_lr': 0.01, 'inner_steps': 3} -> 0.9231
  fold 1 [4/8] {'meta_lr': 0.001, 'inner_lr': 0.01, 'inner_steps': 5} -> 1.0000
  fold 1 [5/8] {'meta_lr': 0.002, 'inner_lr': 0.005, 'inner_steps': 3} -> 0.9231
  fold 1 [6/8] {'meta_lr': 0.002, 'inner_lr': 0.005, 'inner_steps': 5} -> 0.9231
  fold 1 [7/8] {'meta_lr': 0.002, 'inner_lr': 0.01, 'inner_steps': 3} -> 0.9231
  fold 1 [8/8] {'meta_lr': 0.002, 'inner_lr': 0.01, 'inner_steps': 5} -> 0.9231
  fold 1: best {'meta_lr': 0.001, 'inner_lr': 0.01, 'inner_steps': 5} | trainable 8,576,452 | test images also in training 69% | train 1.0000 | TEST 1.0000
  fold 2 [1/8] {'meta_lr': 0.001, 'inner_lr': 0.005, 'inner_steps': 3} -> 1.0000
  fold 2 [2/8] {'meta_lr': 0.001, 'inner_lr': 0.005, 'inner_steps': 5} -> 1.0000
  fold 2 [3/8] {'meta_lr': 0.001, 'inn

100%|██████████| 20.5M/20.5M [00:00<00:00, 115MB/s] 


  fold 1 [1/8] {'lr': 0.001, 'wd': 0.0005, 'episodes': 60} -> 0.8462
  fold 1 [2/8] {'lr': 0.001, 'wd': 0.0005, 'episodes': 100} -> 1.0000
  fold 1 [3/8] {'lr': 0.001, 'wd': 0.001, 'episodes': 60} -> 0.8462
  fold 1 [4/8] {'lr': 0.001, 'wd': 0.001, 'episodes': 100} -> 0.8462
  fold 1 [5/8] {'lr': 0.002, 'wd': 0.0005, 'episodes': 60} -> 0.9231
  fold 1 [6/8] {'lr': 0.002, 'wd': 0.0005, 'episodes': 100} -> 0.9231
  fold 1 [7/8] {'lr': 0.002, 'wd': 0.001, 'episodes': 60} -> 0.9231
  fold 1 [8/8] {'lr': 0.002, 'wd': 0.001, 'episodes': 100} -> 0.9231
  fold 1: best {'lr': 0.001, 'wd': 0.0005, 'episodes': 100} | trainable 790,464 | test images also in training 69% | train 1.0000 | TEST 1.0000
  fold 2 [1/8] {'lr': 0.001, 'wd': 0.0005, 'episodes': 60} -> 0.9231
  fold 2 [2/8] {'lr': 0.001, 'wd': 0.0005, 'episodes': 100} -> 0.9231
  fold 2 [3/8] {'lr': 0.001, 'wd': 0.001, 'episodes': 60} -> 1.0000
  fold 2 [4/8] {'lr': 0.001, 'wd': 0.001, 'episodes': 100} -> 0.9231
  fold 2 [5/8] {'lr': 0.002,